# Silver: CRM products
**Source:** `bronze.crm_prd_info`  ->  **Target:** `silver.crm_products`

**What this notebook does:**
- Remove extra spaces
- Split `prd_key` into **category ID** and **product number**
- Replace missing cost with 0
- Turn product line codes into words
- **Rebuild end dates** (source end dates are wrong: some end before they start)
- Rename columns

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.window import Window

CATALOG = "workspace"

## Read the Bronze table

In [0]:
df = spark.table(f"{CATALOG}.bronze.crm_prd_info")

## 1. Trim spaces

In [0]:
# Remove extra spaces from every text column it exists
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## 2. Split the product key
Example: `CO-RF-FR-R92B-58`  →  category `CO_RF` + product number `FR-R92B-58`.
The category ID must match the ERP category table, which uses `_` instead of `-`.

In [0]:
df = (
    df
    .withColumn("cat_id",  F.regexp_replace(F.substring("prd_key", 1, 5), "-", "_"))
    .withColumn("prd_key", F.substring("prd_key", 7, 100))
)

## 3. Fix cost and product line

In [0]:
df = (
    df
    .withColumn("prd_cost", F.coalesce(F.col("prd_cost"), F.lit(0)))
    .withColumn("prd_line",
        F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
         .when(F.upper(F.col("prd_line")) == "R", "Road")
         .when(F.upper(F.col("prd_line")) == "S", "Other Sales")
         .when(F.upper(F.col("prd_line")) == "T", "Touring")
         .otherwise("n/a"))
)

## 4. Rebuild the dates
A product can have several price versions over time. The source end dates are unreliable, so we rebuild them:
- end date = **day before the next version starts**
- the newest version has **no end date** (it is the current one)

In [0]:
df = df.withColumn("prd_start_dt", F.col("prd_start_dt").cast(DateType()))

versions_in_time = Window.partitionBy("prd_key").orderBy("prd_start_dt")

df = df.withColumn(
    "prd_end_dt",
    F.date_sub(F.lead("prd_start_dt").over(versions_in_time), 1)
)

## 5. Rename columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Write the Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.crm_products")

## Check it Quickly
Every product must have exactly **one** current row (no end date).

In [0]:
result = spark.table(f"{CATALOG}.silver.crm_products")
current = result.filter("end_date IS NULL")
print("all rows:", result.count(), ", current rows:", current.count(),
      ", unique products:", result.select("product_number").distinct().count())
result.display()